In [ ]:
%matplotlib inline

# Notebook 3 — Beam–beam interaction and dynamic aperture

Install 6D beam–beam lenses at the four IPs and compare 200-turn survival with crab sextupoles on and off, keeping beam–beam enabled.

The opposing bunch is fixed (weak–strong model).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xtrack as xt
import xfields as xf
from xfields.config_tools.beambeam_config_tools.config_tools import constant_charge_slicing_gaussian

### Beam parameters

Use the following beam parameters: $N=2.18\times10^{11}$, $\varepsilon_x=0.71$ nm rad, $\varepsilon_y=2.1$ pm rad, and the colliding bunch length $\sigma_z=15.2$ mm.

The full crossing angle is 30 mrad.

In [ ]:
bunch_population = 2.18e11
gemitt_x = 0.71e-9  # m, geometric emittance
gemitt_y = 2.1e-12  # m, geometric emittance
sigma_z = 15.2e-3   # m, colliding bunch length
crossing_angle = 30e-3  # rad, full angle

### Load the nominal optics

Use 6D Twiss with the existing RF cavities.

In [ ]:
env = xt.load('fccee_z.json')
env.vars.load('optics_nominal.json')
line = env.fccee_p_ring
ips = ['ipa', 'ipd', 'ipg', 'ipj']

In [ ]:
# Twiss with bb off
tw0 = line.twiss()
tw0.rows[ips].cols['betx bety']

### Install beam-beam interactions

The transverse covariance matrix describes the beam sizes, momentum spreads and correlations. For the coordinates $u=(x,p_x,y,p_y)$,

$$
\Sigma_{ij}=\langle (u_i-\langle u_i\rangle)(u_j-\langle u_j\rangle)\rangle,
$$

where $\langle\cdot\rangle$ denotes an average over the particles. For a beam centred at zero, for example, $\Sigma_{11}=\langle x^2\rangle$ and $\Sigma_{12}=\langle x p_x\rangle$.

The diagonal terms are the variances:

$$
\Sigma_{11}=\sigma_x^2,\qquad
\Sigma_{22}=\sigma_{p_x}^2,\qquad
\Sigma_{33}=\sigma_y^2,\qquad
\Sigma_{44}=\sigma_{p_y}^2.
$$

Here we assume a waist at each IP, $\alpha_x=\alpha_y=0$, so we set $\Sigma_{12}=\Sigma_{34}=0$. We use the four diagonal terms from the covariance table to describe the opposing beam.

In [ ]:
# Measure beam sizes
sigmas = tw0.get_beam_covariance(gemitt_x=gemitt_x, gemitt_y=gemitt_y)

In [ ]:
# Discretize the strong bunch longitudinally
n_slices = 201

z_centroids, z_cuts, num_particles_per_slice = constant_charge_slicing_gaussian(
    N_part_tot=bunch_population, sigmaz=sigma_z, N_slices=n_slices,
)
z_centroids = z_centroids[::-1]
num_particles_per_slice = num_particles_per_slice[::-1]

### Install the lenses

Use the beam sizes at IPA for the four identical lenses. Xfields takes the half crossing angle.

In [ ]:
bb = xf.BeamBeamBiGaussian3D(
    phi=crossing_angle / 2,  # Half crossing angle
    alpha=0,
    other_beam_q0=-line.particle_ref.q0,
    slices_other_beam_num_particles=num_particles_per_slice,
    slices_other_beam_zeta_center=z_centroids,
    slices_other_beam_Sigma_11=sigmas['Sigma11', 'ipa'],
    slices_other_beam_Sigma_12=0,
    slices_other_beam_Sigma_22=sigmas['Sigma22', 'ipa'],
    slices_other_beam_Sigma_33=sigmas['Sigma33', 'ipa'],
    slices_other_beam_Sigma_34=0,
    slices_other_beam_Sigma_44=sigmas['Sigma44', 'ipa'],
)
env.elements['bb_ipa'] = bb.copy()
env.elements['bb_ipd'] = bb.copy()
env.elements['bb_ipg'] = bb.copy()
env.elements['bb_ipj'] = bb.copy()

# Attach a knob to switch bb on/off
env['on_bb'] = 0
env['bb_ipa'].scale_strength = 'on_bb'
env['bb_ipd'].scale_strength = 'on_bb'
env['bb_ipg'].scale_strength = 'on_bb'
env['bb_ipj'].scale_strength = 'on_bb'

# Insert bb elements in the line
line.insert([
    env.place('bb_ipa', at='ipa'),
    env.place('bb_ipd', at='ipd'),
    env.place('bb_ipg', at='ipg'),
    env.place('bb_ipj', at='ipj'),
])

### Beam–beam tune shift

Compare the low-amplitude tunes with the beam-beam off and on.

In [ ]:
env['on_bb'] = 0
tw_off = line.twiss(chrom=False)
env['on_bb'] = 1
tw_bb = line.twiss(chrom=False)
print('BB off:', tw_off.qx, tw_off.qy)
print('BB on:', tw_bb.qx, tw_bb.qy)

### Add an aperture restriction

Place one aperture limitation at `end_straight_start_ds_ipd` with half-apertures of 9 horizontal and 9 vertical beam sizes from the original optics. The aperture is artificially tight, to allow observing losses on a reduced number of turns.

In [ ]:
aperture_at = 'end_straight_start_ds_ipd'

# Aperture sizes
half_x = 9 * sigmas['sigma_x', aperture_at]
half_y = 9 * sigmas['sigma_y', aperture_at]

# Create aperture
env.elements['loss_ap'] = xt.LimitRect(
    min_x=-half_x, max_x=half_x, min_y=-half_y, max_y=half_y,
)

# Insert aperture
line.insert('loss_ap', at=aperture_at)

### Generate on-momentum particles

Use 16 radii from 0.5 to 8 sigmas and 10 angles in the first quadrant, with zero normalized transverse momenta and `zeta=delta=0`.

In [ ]:
r_values = np.linspace(0.5, 8, 16)
theta_values = np.linspace(0, np.pi / 2, 10)
r_grid, theta_grid = np.meshgrid(r_values, theta_values)
Ax = (r_grid * np.cos(theta_grid)).ravel()
Ay = (r_grid * np.sin(theta_grid)).ravel()

particles_initial = line.build_particles(
    x_norm=Ax, px_norm=0,
    y_norm=Ay, py_norm=0,
    zeta=0, delta=0,
    gemitt_x=gemitt_x, gemitt_y=gemitt_y,
    particle_on_co=tw_bb.particle_on_co,
    W_matrix=tw0['W_matrix', 0],
)

### Beam–beam on, crab sextupoles on

In [ ]:
num_turns = 200  # A realistic DA estimate would need many more turns

env['on_bb'] = ...
p_on = particles_initial.copy()
line.track(p_on, num_turns=num_turns, with_progress=10)

In [ ]:
# Restore particle-ID order, including lost particles, to match the original Ax, Ay grid.
p_on.sort(interleave_lost_particles=True)
survived_on = p_on.state > 0

plt.figure(figsize=(5, 4))
plt.scatter(Ax[survived_on], Ay[survived_on], color='C0', s=15, label='Survived')
plt.scatter(Ax[~survived_on], Ay[~survived_on], color='red', s=55, label='Lost')
plt.title('BB on, crab on')
plt.xlabel(r'$A_x$ [$\sigma_x$]')
plt.ylabel(r'$A_y$ [$\sigma_y$]')
plt.gca().set_aspect('equal')
plt.legend()

### Beam–beam on, crab sextupoles off

In [ ]:
env['on_bb'] = ...
env['cs_frac'] = ...
p_crab_off = particles_initial.copy()
line.track(p_crab_off, num_turns=num_turns, with_progress=10)

In [ ]:
# Restore particle-ID order, including lost particles, to match the original Ax, Ay grid.
p_crab_off.sort(interleave_lost_particles=True)
survived_crab_off = p_crab_off.state > 0

plt.figure(figsize=(5, 4))
plt.scatter(Ax[survived_crab_off], Ay[survived_crab_off], color='C0', s=15, label='Survived')
plt.scatter(Ax[~survived_crab_off], Ay[~survived_crab_off], color='red', s=55, label='Lost')
plt.title('BB on, crab off')
plt.xlabel(r'$A_x$ [$\sigma_x$]')
plt.ylabel(r'$A_y$ [$\sigma_y$]')
plt.gca().set_aspect('equal')
plt.legend()